# 05 - Hyperparameter tuning (validation set only)

Tunes Logistic Regression, KNN, SVM, Decision Tree, Random Forest, XGBoost and LightGBM on all **103 signal features**.

**Rules followed**
- Every setting is **fitted on the training set** and **scored on the validation set**.
- Best setting = highest validation **F1 of the abnormal class (1)**; validation ROC-AUC breaks ties.
- The **test set is not opened** here (see notebook 06).
- Imputer and scaler are inside the pipeline, so they are fitted on training data only.
- All 7 models use the same method. Grids for slow models (SVM, KNN, boosting, Random Forest) are reduced to save run time. Results are saved after each model.

In [1]:
import itertools
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
TARGET_COLUMN = "figo_label"
FEATURE_PREFIXES = ("F_", "U_", "FU_")

# Project root = parent of the notebooks folder
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)

# Load the TRAINING and VALIDATION files only. The test file is NOT opened in this notebook.
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")

# The 103 signal features (IDs and target are excluded)
feature_cols = []
for col in train_df.columns:
    if col.startswith(FEATURE_PREFIXES):
        feature_cols.append(col)
if len(feature_cols) != 103:
    raise ValueError("Expected 103 features, found " + str(len(feature_cols)))

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train_df[TARGET_COLUMN].astype(int)
X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_val = val_df[TARGET_COLUMN].astype(int)
print("Train:", X_train.shape, " Validation:", X_val.shape)

Python 3.13.12 | scikit-learn 1.9.1
Train: (14540, 103)  Validation: (3115, 103)


## Pipeline builder

In [2]:
def make_pipeline(name, params):
    """Imputer (+ StandardScaler for LR, KNN, SVM) + model. Everything is fitted on TRAIN only."""
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if name in ("Logistic Regression", "KNN", "SVM"):
        steps.append(("scaler", StandardScaler()))

    if name == "Logistic Regression":
        model = LogisticRegression(max_iter=3000, random_state=SEED, **params)
    elif name == "KNN":
        model = KNeighborsClassifier(**params)
    elif name == "SVM":
        model = SVC(kernel="rbf", random_state=SEED, **params)
    elif name == "Decision Tree":
        model = DecisionTreeClassifier(random_state=SEED, **params)
    elif name == "Random Forest":
        model = RandomForestClassifier(random_state=SEED, n_jobs=-1, **params)
    elif name == "XGBoost":
        model = XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **params)
    else:
        model = LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, **params)

    steps.append(("model", model))
    return Pipeline(steps)


def grid(options):
    """Turn a dict of lists into a list of every combination (one dict per combination)."""
    keys = list(options.keys())
    combos = []
    for values in itertools.product(*[options[k] for k in keys]):
        combos.append(dict(zip(keys, values)))
    return combos

## Search spaces

In [3]:
# Same method for all 7 models: fit on TRAIN, score on VALIDATION.
# Small models use a full grid. Slow models use a smaller grid or a random sample of settings.
# The order is fast models first and the slow SVM last.
search_space = {
    "Logistic Regression": grid({"C": [0.01, 0.1, 1, 10, 100]}),
    "Decision Tree": grid({"max_depth": [None, 5, 10, 15, 20], "min_samples_leaf": [1, 5, 10, 20],
                           "criterion": ["gini", "entropy"]}),
    "KNN": grid({"n_neighbors": [3, 7, 15, 25], "weights": ["uniform", "distance"], "p": [2]}),
    "LightGBM": list(ParameterSampler({
        "n_estimators": [300, 600], "num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.1],
        "min_child_samples": [10, 20, 40], "subsample": [0.8, 1.0], "colsample_bytree": [0.6, 0.8, 1.0]},
        n_iter=12, random_state=SEED)),
    "XGBoost": list(ParameterSampler({
        "n_estimators": [300, 600], "max_depth": [3, 5, 7], "learning_rate": [0.03, 0.1],
        "subsample": [0.8, 1.0], "colsample_bytree": [0.6, 0.8, 1.0], "min_child_weight": [1, 5]},
        n_iter=12, random_state=SEED)),
    "Random Forest": list(ParameterSampler({
        "n_estimators": [200, 400], "max_depth": [None, 20, 40], "min_samples_leaf": [1, 2, 5],
        "max_features": ["sqrt", 0.3, 0.5], "min_samples_split": [2, 5]},
        n_iter=10, random_state=SEED)),
    "SVM": grid({"C": [1, 10], "gamma": ["scale", 0.01]}),
}
total_settings = 0
for model_name in search_space:
    count = len(search_space[model_name])
    total_settings = total_settings + count
    print(model_name, "->", count, "settings")
print("Total models to train:", total_settings)


Logistic Regression -> 5 settings
Decision Tree -> 40 settings
KNN -> 8 settings
LightGBM -> 12 settings
XGBoost -> 12 settings
Random Forest -> 10 settings
SVM -> 4 settings
Total models to train: 91


## Run the search (fit on train, score on validation)

In [4]:
rows = []
best = {}
RESULTS_DIR.mkdir(exist_ok=True)

for model_name in search_space:
    candidates = search_space[model_name]
    best_key = None
    model_start = time.time()

    for params in candidates:
        start = time.time()

        # Fit on the TRAINING set, score on the VALIDATION set
        pipe = make_pipeline(model_name, params)
        pipe.fit(X_train, y_train)
        val_pred = pipe.predict(X_val)
        if model_name == "SVM":
            val_score = pipe.decision_function(X_val)
        else:
            val_score = pipe.predict_proba(X_val)[:, 1]

        val_f1 = float(f1_score(y_val, val_pred, pos_label=1))
        val_auc = float(roc_auc_score(y_val, val_score))
        rows.append({"model": model_name, "params": json.dumps(params), "val_f1": val_f1,
                     "val_auc": val_auc, "seconds": round(time.time() - start, 1)})

        # Best = highest validation F1; ROC-AUC breaks ties
        key = (val_f1, val_auc)
        if best_key is None or key > best_key:
            best_key = key
            best[model_name] = {"params": params, "val_f1": val_f1, "val_auc": val_auc}

    print(model_name, "| best validation F1 %.4f, AUC %.4f | %.0f s" % (
        best[model_name]["val_f1"], best[model_name]["val_auc"], time.time() - model_start))
    print("   ", best[model_name]["params"])

    # Save after every model so progress is not lost if the run stops
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "tuning_results.csv", index=False)
    (RESULTS_DIR / "best_params.json").write_text(json.dumps(best, indent=1))


Logistic Regression | best validation F1 0.9799, AUC 0.9982 | 2 s
    {'C': 10}
Decision Tree | best validation F1 0.8535, AUC 0.9046 | 41 s
    {'max_depth': 10, 'min_samples_leaf': 20, 'criterion': 'gini'}
KNN | best validation F1 0.8503, AUC 0.9360 | 3 s
    {'n_neighbors': 7, 'weights': 'distance', 'p': 2}
LightGBM | best validation F1 0.9571, AUC 0.9927 | 15 s
    {'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 600, 'min_child_samples': 40, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
XGBoost | best validation F1 0.9579, AUC 0.9923 | 37 s
    {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Random Forest | best validation F1 0.9024, AUC 0.9639 | 104 s
    {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20}
SVM | best validation F1 0.9723, AUC 0.9966 | 18 s
    {'C': 10, 'gamma': 'scale'}


## Earlier Random Forest search (reference only, not used)


In [5]:
# Random Forest is now tuned above with the SAME method as the other models.
# The earlier saved search used 5-fold CV and class_weight=balanced, so it is shown for reference only.
rf_saved = json.loads((RESULTS_DIR / "random_forest_tuned_validation.json").read_text())
print("Earlier CV-based Random Forest validation F1 (reference only):",
      round(rf_saved["validation_metrics"]["abnormal_f1"], 4))
print("Random Forest used in this comparison:", best["Random Forest"]["params"])
print("Random Forest validation F1:", round(best["Random Forest"]["val_f1"], 4))


Earlier CV-based Random Forest validation F1 (reference only): 0.9003
Random Forest used in this comparison: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20}
Random Forest validation F1: 0.9024


## Summary

In [6]:
# Best setting per model, with the default-like validation F1 for comparison
summary_rows = []
for model_name in best:
    summary_rows.append({"model": model_name, "best_val_f1": best[model_name]["val_f1"],
                         "best_val_auc": best[model_name]["val_auc"], "best_params": json.dumps(best[model_name]["params"])})
summary_df = pd.DataFrame(summary_rows).sort_values("best_val_f1", ascending=False)
display(summary_df)

,model,best_val_f1,best_val_auc,best_params
0,Logistic Regression,0.979946,0.998232,"{""C"": 10}"
6,SVM,0.972313,0.996616,"{""C"": 10, ""gamma"": ""scale""}"
4,XGBoost,0.957923,0.992325,"{""subsample"": 0.8, ""n_estimators"": 600, ""min_c..."
3,LightGBM,0.957143,0.992697,"{""subsample"": 0.8, ""num_leaves"": 15, ""n_estima..."
5,Random Forest,0.902410,0.963927,"{""n_estimators"": 200, ""min_samples_split"": 5, ..."
1,Decision Tree,0.853477,0.904580,"{""max_depth"": 10, ""min_samples_leaf"": 20, ""cri..."
2,KNN,0.850318,0.935967,"{""n_neighbors"": 7, ""weights"": ""distance"", ""p"": 2}"


## Save results

In [7]:
# Save the tuning results (validation only) for the final evaluation notebook
pd.DataFrame(rows).to_csv(RESULTS_DIR / "tuning_results.csv", index=False)
(RESULTS_DIR / "best_params.json").write_text(json.dumps(best, indent=1))
print("Saved:", RESULTS_DIR / "tuning_results.csv")
print("Saved:", RESULTS_DIR / "best_params.json")

Saved: c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\tuning_results.csv
Saved: c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\results\best_params.json
